# 02B - Tokenization and Embeddings PyTorch Lab

**Guiding question:** can you trace a pet-sitter's phrase from text to GPT-2 IDs, train only the embedding rows that participated, and prove padding contributes zero loss?

> **What you finished last time:** [02A's notebook](02A-tokenization-and-embeddings-theory.ipynb)
> split our pet notes, fetched the two-number rows for `the ___ sat`, checked
> one-hot selection, and separated static lookup from context mixing.
> **What this notebook delivers:** compact BPE helpers, real GPT-2 `tiktoken` inspection, a controlled `nn.Embedding` learning experiment, and a PyTorch masking proof.
> **What this chapter is not:** a contextual encoder or retrieval application.

We'll reuse the same six pet notes, then make one deliberately smaller learning
game: `cat` and `dog` both support `sat` and `slept`; `bird` supports `flew`
and `sang`. Those six pairs are supplied by us, not secretly mined from the notes.
Their job is to show how lookup rows learn and how one question can have more
than one sensible answer. Turning text into questions belongs to
[03A's notebook](../03-language-models/03A-training-skeleton-and-word-objectives.ipynb).

The setup cell only imports tools, fixes the starting seed, and collects the notes.
It doesn't load pretrained model weights.

For the code-free version, use the [handwritten companion](02B-tokenization-and-embeddings-pytorch-lab-theory.md).


In [ ]:
# -- Setup ------------------------------------------------------------------------
from collections import Counter
import re
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import tiktoken

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})

PET_CORPUS = [
    "the sleepy cat sat on the mat",
    "the dog sat on the rug",
    "the cat slept on the rug",
    "a sleepy dog slept on the mat",
    "the loud bark woke the cat",
    "the rough bark covered the tree",
]
print(f"Corpus sentences: {len(PET_CORPUS)}")


This lab starts from character pieces, counts neighboring pairs, and repeatedly merges a common pair into one reusable piece. The method is called **byte-pair encoding (BPE)**; the toy version keeps every merge visible so you can inspect the rule.

## 1 - A compact BPE you can read in one minute

![BPE loop: count neighbouring pairs, glue the most common pair, repeat. One learned piece, at, is reused by cat, sat and mat.](images/02-02-bpe-glues-common-pairs.png)

*Same loop as 02A, now as three small functions: count pairs, glue the best one, repeat.*


In [ ]:
# -- BPE helpers: initialize, count, and merge -----------------------------------
def initialize_vocabulary(corpus):
    words = Counter(re.findall(r"[a-z-]+", " ".join(corpus).lower()))
    return {tuple(word) + ("</w>",): count for word, count in words.items()}

def count_pairs(vocabulary):
    pairs = Counter()
    for symbols, frequency in vocabulary.items():
        for pair in zip(symbols, symbols[1:]):
            pairs[pair] += frequency
    return pairs

def merge_best_pair(vocabulary, pair):
    updated = {}
    for symbols, frequency in vocabulary.items():
        merged = []
        index = 0
        while index < len(symbols):
            if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
                merged.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                merged.append(symbols[index])
                index += 1
        updated[tuple(merged)] = frequency
    return updated

vocabulary = initialize_vocabulary(PET_CORPUS)
learned_merges = []
for _ in range(20):
    pairs = count_pairs(vocabulary)
    if not pairs:
        break
    pair = pairs.most_common(1)[0][0]
    learned_merges.append(pair)
    vocabulary = merge_best_pair(vocabulary, pair)

print(f"Learned merges: {len(learned_merges)}")
print("First five:", learned_merges[:5])


### Learning the joins is different from using them

The three helpers above split, count, and join. The loop learned an **ordered**
list of joins from the notes. Now freeze that list and replay it on a new word.
A later join may depend on an earlier piece, so sorting the pairs differently
would change the tokenizer.

`apply_merges` scans left to right for each learned pair, carrying untouched
letters through. The unseen `sleepyhead` can reuse the known `sleepy` part and
keep the remaining letters. This character toy assumes those letters are
available; GPT-2's byte alphabet handles wider input coverage.

**Predict:** must an unseen whole word become unknown if smaller pieces remain
available? Check the printed pieces rather than guessing from the word alone.


In [ ]:
# -- Apply learned merges to a new word -------------------------------------------
def apply_merges(word, merges):
    symbols = list(word) + ["</w>"]
    for pair in merges:
        output = []
        index = 0
        while index < len(symbols):
            if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
                output.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                output.append(symbols[index])
                index += 1
        symbols = output
    return symbols

phrase_word = "sleepy"
pieces = apply_merges(phrase_word, learned_merges)
print(f"{phrase_word!r} -> {pieces}")
print(f"Compression: {len(phrase_word) + 1} character symbols -> {len(pieces)} BPE pieces")
print(f"Never-seen word: 'sleepyhead' -> {apply_merges('sleepyhead', learned_merges)}")


### Your turn - change the merge budget

**Your turn:** change only the number of learned merges and predict the target piece count.


In [ ]:
# -- Your turn: one-variable BPE budget experiment -------------------------------
# CHANGE THIS: try 5 or 12
merge_budget = 8
tested_pieces = apply_merges("sleepy", learned_merges[:merge_budget])
print(f"Budget {merge_budget}: {tested_pieces} ({len(tested_pieces)} pieces)")
print("Check: using more already-learned merges cannot increase this word's piece count.")


## 2 - Inspect the production GPT-2 tokenizer

The toy tokenizer uses characters. GPT-2 starts from bytes and applies a pretrained merge table.

**Predict:** will `" cat"` and `"cat"` necessarily receive the same IDs? GPT-2 tokens can include leading whitespace.


In [ ]:
# -- GPT-2 tiktoken: IDs, bytes, whitespace, and round-trip -----------------------
encoder = tiktoken.get_encoding("gpt2")
phrase = "The sleepy cat sat on the mat."
token_ids = encoder.encode(phrase)
token_bytes = [encoder.decode_single_token_bytes(token_id) for token_id in token_ids]
decoded = encoder.decode(token_ids)

print(f"Text: {phrase!r}")
print(f"Token IDs: {token_ids}")
print(f"Token bytes: {token_bytes}")
print(f"Round-trip exact: {decoded == phrase}")
print(f"' cat' IDs: {encoder.encode(' cat')}")
print(f"'cat' IDs:  {encoder.encode('cat')}")
assert decoded == phrase


### That round-trip worked. How much space did the notes use?

The phrase came back exactly, even though some pieces carry leading spaces.
Now encode all six notes with newline separators and count their positions.
Characters per token is a compression check on **this text**, not a universal
score for the tokenizer. Spelling, punctuation, and language can change it.

**Predict:** will our entire note collection use fewer token positions than
characters? The next cell measures it with the same pretrained tokenizer;
it doesn't learn new merges or fetch model weights.


In [ ]:
# -- Compression evidence on the pet corpus --------------------------------------
pet_text = "\n".join(PET_CORPUS)
production_ids = encoder.encode(pet_text)
characters_per_token = len(pet_text) / len(production_ids)
print(f"Characters: {len(pet_text)}")
print(f"GPT-2 tokens: {len(production_ids)}")
print(f"Characters per token: {characters_per_token:.2f}")
print("Boundary: this measures this corpus, not every language or kind of writing.")


A text piece first gets an integer address. The model uses that address to fetch a row of adjustable numbers, and the prediction task supplies the pressure that can teach the row. This lookup row is an **embedding**; the next experiment checks exactly which rows receive a learning signal.

### One tiny lookup before we train anything

Put GPT-2's vocabulary aside for a moment. We are changing to a **local teaching
vocabulary**, not copying GPT-2's rows or reusing its ID numbers.

Repeat 02A's exact table: `the` is ID 0 with `[2, 0]`, `cat` is ID 1
with `[1, 3]`, and `sat` is ID 2 with `[0, 4]`. Fetch the rows for
`the ___ sat`. Two visible words should give two rows, each two numbers wide.
The covered `cat` is not another input.

The explicit one-hot route is only a cross-check. Real lookup needn't construct
all those zeros. These hand-picked numbers still aren't learned word meaning.

**Predict:** should PyTorch fetch the same rows NumPy did? And should ID zero
fetch zeros, or the row stored at address zero?


In [ ]:
local_words = ["the", "cat", "sat"]
local_table = torch.tensor([[2.0, 0.0], [1.0, 3.0], [0.0, 4.0]])
visible_word_ids = torch.tensor([0, 2], dtype=torch.long)
# IDs [2] select rows from [3,2] -> [2,2], one row per visible word.
local_rows = nn.functional.embedding(visible_word_ids, local_table)
# [2] -> [2,3] expands IDs into checklists; use the table's dtype for multiplication.
one_hot_rows = nn.functional.one_hot(visible_word_ids, num_classes=3).to(local_table.dtype)
# [2,3] @ [3,2] -> [2,2], the same selected rows without any learned transformation.
explicit_rows = one_hot_rows @ local_table
print("Visible words:", [local_words[index] for index in visible_word_ids.tolist()])
print("Fetched rows:\n", local_rows)
print("One-hot route matches:", torch.equal(local_rows, explicit_rows))
assert torch.equal(local_rows, torch.tensor([[2.0, 0.0], [0.0, 4.0]]))
assert torch.equal(local_rows, explicit_rows)
print("Takeaway: lookup selects the stored numbers; the next experiment will actually learn them.")


## 3 - Random rows can identify words. Can we make them more useful?

First, a possible mix-up: the GPT-2 tokenizer above supplied pieces, IDs, and decoding rules.
It did **not** supply GPT-2's embedding weights. `nn.Embedding(...)` below creates a fresh
random table, and `nn.Linear(...)` creates a fresh prediction layer. The seed makes the
starting draw repeatable; it doesn't make it pretrained.

Here are the little patterns we want the model to learn:

```text
cat  -> sat, slept
dog  -> sat, slept
bird -> flew, sang
```

`cat` and `dog` have different name tags, but this guessing game asks them to
support the same answers: both sit, both sleep. Random rows don't deliberately express that shared job.
We could keep them fixed and make the predictor learn around them. Instead, let both parts
adapt: the rows can become useful features, and the prediction layer can learn how to read
those features. `model.parameters()` includes both sets of weights.

### Are we changing the input, or changing the weights?

Both phrases can sound right until you separate two objects:

```text
fixed cat ID -> fetch cat's stored row -> predictor -> word scores
                        ^                    ^
                        |---- weights -------|
```

We pass the **integer ID** into the model. The embedding layer fetches a row
that started random but is itself part of the model's trainable weights.
Backpropagation calculates feedback for that row; the optimizer changes its
stored numbers. Next time, the **same ID** fetches the **updated row**.
We don't change the text, train the integer, or draw a new random row per use.

The row is an input to the prediction layer, but a weight of the whole model.
That is how a model can learn its own input features without editing its data.

Why bother? Useful shared features let one set of prediction weights reuse a pattern across
different words, rather than treating every word as an unrelated case. That is the link to
**semantic similarity**: learning that helps with `cat` may also help with `dog`
because they share useful features, not because we assigned them the same ID. Their rows
stay separate. Similar usage can also connect words that aren't synonyms, including opposites;
an embedding is a learned relationship, not a certificate that two words mean the same thing.
That is a reason to
try learning embeddings, not a promise that this tiny run generalizes. A strong predictor
can also work with frozen random rows; pretrained rows are another option when available.

**Watch the measurements:** loss tells us whether these guesses improved; the before/after
distance tells us how two rows moved. Shared targets do not require their full vectors to
become close: the output layer may ignore some differences. We only assert that loss falls,
not that every pair must move closer or that training beats frozen embeddings.

![Opening the cat drawer fetches its row; the predictor makes a shortlist of sat, slept, flew and sang. Comparing with the answer sat updates only the cat row and the predictor weights; dog and bird rows are unchanged.](images/02-05-practice-nudges-rows.png)

*One wrong guess nudges the predictor and the one row it fetched. Rows that weren't opened stay put.*


In [ ]:
# -- Controlled corpus and nn.Embedding model ------------------------------------
lessons = [
    ("cat", "sat"),
    ("dog", "sat"),
    ("cat", "slept"),
    ("dog", "slept"),
    ("bird", "flew"),
    ("bird", "sang"),
]
words = sorted({word for pair in lessons for word in pair})
word_to_id = {word: index for index, word in enumerate(words)}
input_ids = torch.tensor([word_to_id[source] for source, _ in lessons], dtype=torch.long)
target_ids = torch.tensor([word_to_id[target] for _, target in lessons], dtype=torch.long)

class LookupPredictor(nn.Module):
    def __init__(self, vocabulary_size, dimension=4):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, dimension)
        self.head = nn.Linear(dimension, vocabulary_size, bias=False)

    def forward(self, ids):
        # IDs [B] -> learned rows [B,D] -> word scores [B,V]; one D -> V projection is reused for every row.
        return self.head(self.embedding(ids))

model = LookupPredictor(len(words))
before = model.embedding.weight.detach().clone()
optimizer = torch.optim.Adam(model.parameters(), lr=0.08)
losses = []

for _ in range(250):
    logits = model(input_ids)
    loss = nn.functional.cross_entropy(logits, target_ids)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))

after = model.embedding.weight.detach().clone()
cat_id = word_to_id["cat"]
dog_id = word_to_id["dog"]
before_distance = torch.dist(before[cat_id], before[dog_id]).item()
after_distance = torch.dist(after[cat_id], after[dog_id]).item()

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
print(f"cat/dog distance: {before_distance:.4f} -> {after_distance:.4f}")
print("Takeaway: shared prediction pressure changed the addressed rows.")
assert losses[-1] < losses[0]


### The loss fell. Does the cat have to pick one answer?

Read the model as two small jobs. `nn.Embedding` fetches a four-number row;
`nn.Linear` turns it into one raw score per possible answer. The loop compares
those scores with the supplied answer and adjusts **both** sets of weights.
The distance measurement describes how rows moved, not a test of word meaning.

Our questions contain a deliberate ambiguity: the exact same `cat` input is
paired once with `sat` and once with `slept`. A deterministic forward pass
cannot know which card we drew. The useful answer is a spread of chances,
roughly half for each, not certainty about both at once.

**Softmax** turns raw scores into positive chances that add up to one.
`argmax` always chooses the largest. **Sampling** draws from the whole spread:
likely answers appear more often, but a second answer can still appear.
Embeddings don't supply this randomness; sampling does.

**Predict:** with the same `cat` input, should the chance table change between
draws? Should every sampled answer have to be the same?


In [ ]:
model.eval()
with torch.inference_mode():
    animal_ids = torch.tensor([word_to_id["cat"], word_to_id["dog"]], dtype=torch.long)
    # Scores [2,V] -> probabilities [2,V]: normalize across possible words, not animals.
    animal_chances = model(animal_ids).softmax(dim=-1)
    for animal, chances in zip(["cat", "dog"], animal_chances):
        # One [V] row -> its two highest chances and their word addresses [2].
        top_chances, top_ids = chances.topk(2)
        shortlist = [(words[index], round(chance, 4)) for index, chance in zip(top_ids.tolist(), top_chances.tolist())]
        print(f"{animal} shortlist:", shortlist)
        assert {word for word, _ in shortlist} == {"sat", "slept"}

    cat_chances = animal_chances[0]
    # [V] -> 12 sampled addresses; replacement lets an answer appear more than once.
    sampled_ids = torch.multinomial(
        cat_chances, num_samples=12, replacement=True,
        generator=torch.Generator().manual_seed(23),
    )
    sampled_words = [words[index] for index in sampled_ids.tolist()]
    print("Always choosing the largest:", words[cat_chances.argmax().item()])
    print("Drawing twelve times:", sampled_words)
    assert torch.allclose(animal_chances.sum(dim=1), torch.ones(2))
    assert len(set(sampled_words)) > 1
print("Takeaway: one input gives a fixed spread of chances; sampling can produce different answers.")


### Which rows got that feedback?

The full training run opened `cat`, `dog`, and `bird` repeatedly. Now start a
**fresh** model and ask only `cat -> sat` once. Save the IDs and the stored
weights, work backward, then take one optimizer step.

Only the `cat` **input embedding row** should get a gradient. `sat` is the
correct output label, not another lookup input. The separate prediction layer
also receives feedback. We check both gradients **and actual weight changes**:
`backward()` calculates the feedback; `step()` applies it.

This close-up uses plain SGD with no momentum or weight decay, so unused rows
stay exactly unchanged. An optimizer carrying momentum from older batches or
applying weight decay can also move a row for those reasons.

**Predict:** after `step()`, should the cat ID change, its stored row change,
or both? Would the next lookup at the same ID fetch the old row or the new one?


In [ ]:
# -- One lesson changes a stored row, not the integer that selects it ------------
probe_model = LookupPredictor(len(words))
probe_input_ids = torch.tensor([word_to_id["cat"]], dtype=torch.long)
probe_target_ids = torch.tensor([word_to_id["sat"]], dtype=torch.long)
saved_ids = probe_input_ids.clone()
saved_targets = probe_target_ids.clone()
saved_rows = probe_model.embedding.weight.detach().clone()
saved_head = probe_model.head.weight.detach().clone()
probe_optimizer = torch.optim.SGD(probe_model.parameters(), lr=0.1)

probe_loss = nn.functional.cross_entropy(probe_model(probe_input_ids), probe_target_ids)
probe_loss.backward()
row_gradient_norms = probe_model.embedding.weight.grad.norm(dim=1)
nonzero_rows = [words[index] for index in torch.nonzero(row_gradient_norms).flatten().tolist()]
probe_optimizer.step()

changed = (probe_model.embedding.weight.detach() != saved_rows).any(dim=1)
changed_words = [words[index] for index in changed.nonzero().flatten().tolist()]
assert nonzero_rows == changed_words == ["cat"]
assert torch.equal(probe_input_ids, saved_ids) and torch.equal(probe_target_ids, saved_targets)
assert not probe_input_ids.requires_grad and probe_input_ids.grad is None
assert not torch.equal(probe_model.head.weight.detach(), saved_head)
assert torch.equal(
    probe_model.embedding(probe_input_ids).detach(),
    probe_model.embedding.weight.detach()[saved_ids],
)
print("Input ID before / after:", saved_ids.tolist(), probe_input_ids.tolist())
print("Cat row before:", saved_rows[word_to_id["cat"]].tolist())
print("Cat row after: ", probe_model.embedding(probe_input_ids).detach()[0].tolist())
print("Embedding rows with gradients / changes:", nonzero_rows, changed_words)
print("Predictor weights changed too; input IDs and target IDs stayed fixed.")
print("Takeaway: the same address now fetches an updated row. The data wasn't edited.")


The snapshots separate the address from its contents: the integer ID is unchanged,
but a second lookup at that ID fetches the updated cat row. The `sat` target stayed
fixed too; checking an answer did not turn it into another input.

Short examples need empty `<pad>` slots to fit beside longer examples in one rectangle. Those slots are not real words, so a correct training score must ignore them rather than reward the model for predicting blanks.

## 4 - Pad the batch, then prove padding contributes zero loss

![A three-row batch with six real words and three blank seats. Averaging all nine cells is wrong; masking the blanks or using ignore_index both give the same correct answer.](images/02-06-two-ways-to-skip-padding.png)

*Two different routes, one honest answer. The next cells check that they really match.*


In [ ]:
# -- Padded batch and mask visualization -----------------------------------------
PAD = "<pad>"
batch_tokens = [
    ["the", "cat", "sat"],
    ["the", "dog"],
    ["bird"],
]
batch_vocab = [PAD] + sorted({token for row in batch_tokens for token in row})
batch_to_id = {token: index for index, token in enumerate(batch_vocab)}
pad_id = batch_to_id[PAD]
max_length = max(map(len, batch_tokens))
# Unequal word lists become [B,T_max] IDs; fill real prefixes and leave the remaining seats as pad.
padded = torch.full((len(batch_tokens), max_length), pad_id, dtype=torch.long)
for row_index, row in enumerate(batch_tokens):
    padded[row_index, : len(row)] = torch.tensor([batch_to_id[token] for token in row])
mask = padded.ne(pad_id)

fig, ax = plt.subplots(figsize=(8, 3))
image = ax.imshow(mask.numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set(xticks=range(max_length), yticks=range(len(batch_tokens)))
ax.set(xlabel="Position", ylabel="Sequence", title="PyTorch real-token mask")
plt.colorbar(image, ax=ax)
plt.show()
print(padded)
print(mask)


### A rectangle is useful. Grading every seat is not.

The previous cell built three rows of different lengths, then filled spare
seats with `<pad>`. Its Boolean mask says which positions are real words.
Now give each seat a made-up score vector. We are testing loss bookkeeping,
not asking an untrained model to understand those sentences.

Cross-entropy expects one score row beside each answer. We flatten example
and position together in the **same order**, keeping the vocabulary axis.
Then compare two reductions: average all seats, or only the six real targets.
`ignore_index` should agree with the explicit real-target calculation.

**Predict:** does excluding pad targets change only the numerator, or must the
number of graded targets change too?


In [ ]:
# -- Wrong versus correct PyTorch reductions -------------------------------------
torch.manual_seed(7)
vocabulary_size = len(batch_vocab)
# Add one vocabulary axis: targets [B,T] need logits [B,T,V], a score for every possible answer.
logits = torch.randn(*padded.shape, vocabulary_size)
targets = padded.clone()
# Merge B and T in matching order: [B,T,V] -> [B*T,V], targets [B,T] -> [B*T].
per_position = nn.functional.cross_entropy(
    logits.reshape(-1, vocabulary_size),
    targets.reshape(-1),
    reduction="none",
)
# Keep the same flattened answer order for the Boolean real-token mask [B*T].
flat_targets = targets.reshape(-1)
real = flat_targets.ne(pad_id)

# [B*T] -> scalar grades blanks too; the next reduction keeps [N_real] entries and divides by N_real.
wrong_loss = per_position.mean()
explicit_loss = per_position[real].sum() / real.sum()
ignore_index_loss = nn.functional.cross_entropy(
    logits.reshape(-1, vocabulary_size),
    flat_targets,
    ignore_index=pad_id,
)

print(f"Wrong all-cell mean: {wrong_loss.item():.6f}")
print(f"Explicit masked mean: {explicit_loss.item():.6f}")
print(f"ignore_index mean:    {ignore_index_loss.item():.6f}")
print(f"Correct methods match: {torch.allclose(explicit_loss, ignore_index_loss)}")
assert torch.allclose(explicit_loss, ignore_index_loss)


### Ignoring pad answers is not the same as protecting the pad row

`ignore_index` controls **which answers count toward loss**. It doesn't configure
the input lookup table. `padding_idx` separately tells an embedding which
input row should receive no gradient from lookup.

The next cell sums fetched values only to send an easy-to-inspect probe gradient
through them. It is not another language-learning objective. Real word rows
should receive feedback, while the pad row should not.

**Predict:** if we set `padding_idx`, can we stop masking output targets?
No: one setting protects an input row, the other decides what gets graded.


In [ ]:
# -- Padding-row contract: padding_idx prevents row updates ----------------------
embedding = nn.Embedding(len(batch_vocab), 5, padding_idx=pad_id)
# IDs [B,T] -> rows [B,T,5]; summing all features -> scalar sends a probe gradient through each lookup.
vectors = embedding(padded)
vectors.sum().backward()
pad_gradient = embedding.weight.grad[pad_id].abs().max().item()
real_gradient = embedding.weight.grad[batch_to_id["cat"]].norm().item()
print(f"Maximum pad-row gradient: {pad_gradient:.6f}")
print(f"Cat-row gradient norm: {real_gradient:.6f}")
assert pad_gradient == 0.0
assert real_gradient > 0.0


### Your turn - add padding without changing the answer

**Your turn:** append one extra pad column to both logits and targets. The correct loss should remain unchanged; the wrong mean usually will not.


In [ ]:
# -- Your turn: padding invariance of the correct objective -----------------------
# CHANGE THIS: try 1 or 3 extra pad columns
extra_pad_columns = 2
extra_logits = torch.randn(len(batch_tokens), extra_pad_columns, vocabulary_size)
# Append time slots, not examples: [B,T,V] + [B,E,V] -> [B,T+E,V].
extended_logits = torch.cat([logits, extra_logits], dim=1)
# Give those same extra slots pad answers: [B,T] + [B,E] -> [B,T+E].
extended_targets = torch.cat(
    [targets, torch.full((len(batch_tokens), extra_pad_columns), pad_id, dtype=torch.long)],
    dim=1,
)
# Flatten the enlarged grids together: [B,T+E,V] -> [B*(T+E),V], answers -> [B*(T+E)].
extended_loss = nn.functional.cross_entropy(
    extended_logits.reshape(-1, vocabulary_size),
    extended_targets.reshape(-1),
    ignore_index=pad_id,
)
print(f"Original correct loss: {ignore_index_loss.item():.6f}")
print(f"Extended correct loss: {extended_loss.item():.6f}")
print(f"Padding invariant: {torch.allclose(ignore_index_loss, extended_loss)}")
assert torch.allclose(ignore_index_loss, extended_loss)


## 5 - Evidence scorecard and the next question

| Contract | Measured evidence |
|---|---|
| BPE merge budget | `sleepy`: 7 symbols to 1 piece after 20 merges; 5 pieces at budget 8 |
| production tokenizer | 166 characters, 43 GPT-2 tokens, 3.86 characters per token; exact round-trip; ` cat` gives `[3797]`, `cat` gives `[9246]` |
| IDs are addresses | NumPy's three-word table and PyTorch lookup agree; one-hot multiplication selects the same rows |
| A guess can be a shortlist | `cat` and `dog`: about 50% each for `sat` and `slept`; twelve cat draws include both, without retraining |
| Feedback reaches fetched rows | only the `cat` input row gets a gradient and changes after the fresh SGD step; input/target IDs stay fixed; the predictor changes too |
| rows learn from use | loss 2.0201 to 0.6932; cat/dog distance 3.2438 to 3.0296 |
| padding is not evidence | explicit reduction equals `ignore_index` |
| padding row stays inert | `padding_idx` row gradient is zero |

**Built and measured:** BPE, GPT-2 inspection, one-hot versus compact lookup, trainable rows with fixed IDs, probability shortlists and sampling, padded batch, loss masking.

**Explained or illustrated:** static versus contextual boundary.

**Named and out of scope:** Word2Vec/GloVe history (optional branch), semantic search (RAG), and causal attention masking (Transformers).

You now have token IDs, compact rows, a learned shortlist, batch masks, and
loss masks kept apart. The three-word table and these six practice pairs are
the exact examples 03A will recall. Its real training vocabulary will be larger,
and its main cat probe is **the sleepy cat sat on the mat**; the local IDs here
must not be carried into that new vocabulary. But we handed this model its six practice pairs. [03A](../03-language-models/03A-training-skeleton-and-word-objectives.ipynb) learns from short stories, keeps one familiar cat sentence for close-ups, and asks: can ordinary text write the practice questions for us?
